# Calibration and uncertainty propagation

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/20-calibration.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Up until now we have been creating models that may accurately represent a local
epidemic, but (at best) only provide one possible epidemic profile consistent
with observations. Here we obtain a **range** of parameter values and epidemic
trajectories consistent with those observations, and thereby quantify uncertainty.

The source chapter implemented Metropolis by hand. In summer4 the same ideas
are the `BayesianModel` workflow: named priors, a likelihood on targets, MCMC
via numpyro, then `posterior_runs` to project the epidemic with uncertainty.


In [ ]:
from typing import NamedTuple

import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    OutputSet,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
    derived_refs,
)
from summer4.epi.calibration import BayesianModel, NormalLikelihood, Uniform

AXIS_I = {"index": "time", "value": "number infectious"}


## Calibration data

Dummy observations that look loosely like a bell — plausible active-case
counts through an epidemic wave.


In [ ]:
data = pd.DataFrame(
    {
        "active_cases": {
            60.0: 3000.0,
            80.0: 8500.0,
            100.0: 21000.0,
            120.0: 40000.0,
            140.0: 44000.0,
            160.0: 30000.0,
            180.0: 16000.0,
            200.0: 7000.0,
        }
    }
)
figure = data["active_cases"].plot(kind="scatter", title="Observed active cases")
figure.update_layout(xaxis_title="time", yaxis_title="number infectious")
assert len(data) == 8


## SIR model, priors, and likelihood

A frequency-free contact SIR with a Uniform prior on the infection rate and a
Normal likelihood on the observed infecteds series (sd set from a loose
tolerance on the data scale).


In [ ]:
class Rates(NamedTuple):
    infection: float
    recovery: float


state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
refs = derived_refs(Rates)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], refs.infection))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], refs.recovery))
cm = model.compile()

y0 = PropertyData.wrap(pmap, np.array([1.0e5 - 10.0, 10.0, 0.0]))
obs_times = np.asarray(data.index, dtype=float)
obs_values = np.asarray(data["active_cases"].to_numpy(), dtype=float)
qty = Compartments(where=state["I"])
sd = float(np.mean(np.abs(obs_values)) * 0.15)

targets = TargetSet(
    targets=(
        Target(
            key="I",
            times=obs_times,
            values=obs_values,
            quantity=qty,
            likelihood=NormalLikelihood(sd=sd),
        ),
    )
)
outputs = OutputSet()
outputs["I"] = Compartments(where=state["I"]).total()

bm = BayesianModel(
    cm,
    {"infection": 0.2, "recovery": 0.1},
    priors=(Uniform("infection", 0.05, 0.5),),
    targets=targets,
    outputs=outputs,
    y0=y0,
    run_kwargs=dict(t0=0.0, t1=250.0, dt=1.0, solver="euler"),
)


## MAP and MCMC

`find_map` maximises the joint density; `sample` draws a NUTS posterior. The
MAP infection rate should sit near the posterior median.


In [ ]:
mapped = bm.find_map(steps=120, seed=0)
idata = bm.sample(
    kind="nuts",
    num_warmup=80,
    num_samples=80,
    num_chains=1,
    seed=1,
    progress_bar=False,
)
post = np.asarray(idata.posterior["infection"]).reshape(-1)
median = float(np.median(post))
print(f"MAP={float(mapped['infection']):.3f}, posterior median={median:.3f}")

hist = pd.DataFrame({"infection": post})
figure = hist.plot.hist(nbins=16, title="Posterior infection rate")
figure.update_layout(xaxis_title="infection", yaxis_title="count", showlegend=False)
figure.add_vline(x=float(mapped["infection"]), line_dash="dash", annotation_text="MAP")

assert abs(float(mapped["infection"]) / median - 1.0) < 0.35


## Uncertainty in trajectories

`posterior_runs` evaluates a subsample of draws and returns quantile ribbons
in the same frame schema the tuberculosis analyses write to parquet. The
observed points should sit inside the 95% ribbon near the peak.


In [ ]:
runs = bm.posterior_runs(idata, n=30, seed=2, batch_size=10)
ribbons = runs.quantiles(q=(0.025, 0.5, 0.975))["baseline"]
band = pd.DataFrame(
    {
        "low": ribbons[("I", "0.025")].to_numpy(),
        "median": ribbons[("I", "0.5")].to_numpy(),
        "high": ribbons[("I", "0.975")].to_numpy(),
    },
    index=np.asarray(ribbons.index),
)
figure = band.plot(title="Posterior infecteds ribbon")
figure.update_layout(xaxis_title="time", yaxis_title="number infectious")
figure.add_scatter(
    x=obs_times,
    y=obs_values,
    mode="markers",
    name="observed",
)

# Dummy data is deliberately rough; claim the ribbon is ordered and the
# median trajectory peaks in the same decade as the observations.
assert (band["high"] >= band["median"]).all() and (band["median"] >= band["low"]).all()
assert float(band["median"].max()) > float(band["median"].iloc[0])
